In [5]:
from pathlib import Path 
from pypdf import PdfReader

pdf_path = Path("../data/samplerag.pdf")
reader = PdfReader(pdf_path)

print(f"Print the number of pages: {len(reader.pages)}")

Print the number of pages: 23


In [6]:
all_text=[]

for index, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        all_text.append(text)
    else:
        print(f"{index+1} returned no text")

raw_text="\n".join(all_text)
print(raw_text[:2000])

Think of RAG as: 
“Let the model look things up before it answers.” 
A beginner-friendly one-line definition: 
Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves 
relevant information from a knowledge base, then uses that information to generate a 
grounded answer. 
The key intuition is this: 
• A normal LLM answers from what it learned during training 
• A RAG system answers from both 
o the model’s general knowledge 
o fresh, specific, private, or domain-specific documents retrieved at query 
time 
So instead of asking the model to “remember everything,” you give it a search step. 
 
The simplest mental model 
RAG is like an open-book exam. 
Without RAG: 
• the model answers from memory 
With RAG: 
• the model first opens the right pages of the book 
• then answers using those pages 
That is the whole idea. 
 
Your flow, cleaned up 
You wrote: 
user -> question -> embedding model -> vector -> similarity search -> (vector db -> similar 
passages) -> s

In [10]:
import re

def cleaning_string(text:str)->str:
    text = text.replace("\x00"," ")
    text = re.sub(r"\s+"," ",text)
    return text.strip()

cleaned_text = cleaning_string(raw_text)
print(cleaned_text[:2000])

Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific documents retrieved at query time So instead of asking the model to “remember everything,” you give it a search step. The simplest mental model RAG is like an open-book exam. Without RAG: • the model answers from memory With RAG: • the model first opens the right pages of the book • then answers using those pages That is the whole idea. Your flow, cleaned up You wrote: user -> question -> embedding model -> vector -> similarity search -> (vector db -> similar passages) -> semantic search context retri

In [11]:
def chunk_text(text:str, chunk_size:int, overlap:int):
    chunks=[]
    start = 0
    text_length=len(cleaned_text)
    while start < text_length:
        end = min(start+chunk_size,text_length)
        chunks.append(cleaned_text[start:end])
        if end == text_length:
            break
        start+=chunk_size-overlap
    return chunks

chunks = chunk_text(cleaned_text,500,100)
print(f"Total length of chunks:{len(chunks)}")
print("\nFirst Chunk\n")
print(chunks[0])

Total length of chunks:49

First Chunk

Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific docume


In [12]:
for i in range(min(3,len(chunks))):
    print(f"\nCHUNK-{i}\n")
    print(chunks[i])


CHUNK-0

Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific docume

CHUNK-1

wers from both o the model’s general knowledge o fresh, specific, private, or domain-specific documents retrieved at query time So instead of asking the model to “remember everything,” you give it a search step. The simplest mental model RAG is like an open-book exam. Without RAG: • the model answers from memory With RAG: • the model first opens the right pages of the book • then answers using those pages That is the whole idea. Your flow, cleaned up You wrote: user -> quest

In [13]:
chunk_metadata=[]

for i,chunk in enumerate(chunks):
    chunk_metadata.append({"chunk_id":i,
                           "text":chunk,
                           "source":"samplerag.pdf",
                           "length":len(chunk)})

chunk_metadata[0]

{'chunk_id': 0,
 'text': 'Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific docume',
 'source': 'samplerag.pdf',
 'length': 500}